# Session 1 — Exercises

Four of them, tagged by difficulty. **Do them before Saturday** — exercise 1 is the one that has to
work before session 2 can run at all, and exercise 3 is the scaffold session 2 builds on.

| | Exercise | What it's really teaching |
|---|---|---|
| ⭐ | Your environment, proved | a green checker, a real call, and a deliberate failure read calmly |
| ⭐ | Token detective | measure, don't assume |
| ⭐⭐ | Ticket loader | the dict-update pattern that returns in S10 |
| ⭐⭐⭐ | Survey the field | choosing a model deliberately — no code, three browser tabs |

Then three **interview questions** at the bottom. Write your answers before looking at
`solutions.ipynb` — reading a solution feels like learning and isn't.

**Rules of engagement:** getting stuck is the point; getting stuck for an hour is not. Twenty minutes
on one TODO, then ask.


## Setup

Run this once. If it fails, run `uv run python session-01/check_setup.py` from the repo root — it tells you exactly
what to fix.

In [ ]:
import os
from pathlib import Path

import yaml
from dotenv import load_dotenv
from google import genai

load_dotenv(Path.cwd().parent / ".env")
client = genai.Client()
MODEL = "gemini-3.1-flash-lite"

DATA = Path.cwd().parent / "data"
tickets = yaml.safe_load((DATA / "tickets.yml").read_text(encoding="utf-8"))

print(f"key found: {bool(os.getenv('GEMINI_API_KEY'))} · {len(tickets)} tickets · model {MODEL}")

---
## ⭐ Exercise 1 — Your environment, proved

Everything else this term assumes this works. Session 2 does not stop to fix laptops, so prove it
tonight rather than on Saturday.

### Part A — the checker

In a terminal, from the repo root:

```bash
uv run python session-01/check_setup.py
```

Every line should read PASS. If one doesn't, the line ends with the fix — do the **first** failure,
run it again, and note here what it was and what fixed it. (Writing it down is the exercise: you will
hit the same thing on another machine one day.)

> **What failed, and what fixed it:** …

### Part B — a real call, and what it cost

Fill in the two TODOs below. When it prints a reply and three token counts, your environment is done.


In [ ]:
# The setup cell above already built `client` and `MODEL`.

TICKET = "My headphones arrived with a torn earcup pad. I've only had them three days."

# TODO 1: send TICKET to the model, asking for a one-sentence reply to the customer.
#         `client.interactions.create(model=..., input=...)` — section 3 of the lesson notebook.
interaction = None

# TODO 2: print the reply text, then the three numbers on `interaction.usage`.
#         Section 7.2 shows what they are called.
if interaction is None:
    print("TODO 1 not done yet")
else:
    print("reply :", ...)
    print("usage :", ...)


### Part C — break it on purpose

A key that works is not the same as knowing what a broken one looks like. Run **one** of these
deliberately, read the error, and write down the single line that tells you what is wrong:

- a model ID that does not exist (`"gemini-does-not-exist"`), or
- an empty API key (`genai.Client(api_key="")`).

The lesson notebook's `explain_error` helper (section 1.3) exists for exactly this. An API error is a
wall of JSON with one useful line in it; finding that line quickly is a skill worth ten minutes now.

> **What I broke:** …
>
> **The line that told me:** …


---
## ⭐ Exercise 2 — Token detective

Cost, speed and memory are all counted in **tokens**, so knowing what's expensive is a real
engineering skill rather than trivia.

### Part A — predict, in writing, before you run anything

Three texts of roughly equal length: English prose, Bangla prose, Python code. **Which costs the most
tokens per character?**

Write your prediction in the `prediction` variable below, and one sentence on *why*. Committing to a
guess before measuring is the whole exercise — it's how you find out what you believe.

> **My prediction:** …
>
> **Because:** …

### Part B — now measure

Fill in the `count_tokens` call. The API is
`client.models.count_tokens(model=MODEL, contents=text).total_tokens`.

In [ ]:
samples = {
    "English": "Where is my order? I ordered headphones last week and the tracking has not moved.",
    "Bangla":  "আমার অর্ডার কোথায়? আমি গত সপ্তাহে হেডফোন অর্ডার করেছি এবং ট্র্যাকিং আপডেট হয়নি।",
    "Python":  "def triage(text: str) -> dict:\n    return {'category': 'refund', 'urgency': 3}",
}

prediction = "???"          # <- "English", "Bangla" or "Python"
print(f"my prediction for most tokens per character: {prediction}\n")

print(f"{'':10} {'tokens':>7} {'chars':>7} {'chars/token':>13}")
for name, text in samples.items():
    n = None                # TODO: count the tokens in `text`
    if n is None:
        print(f"{name:10} {'TODO':>7}")
        continue
    print(f"{name:10} {n:>7} {len(text):>7} {len(text) / n:>13.2f}")

### Part C — explain the result

Were you right? Whether or not you were, answer these:

1. Which text was most expensive per character, and what is it about that text that fragments into
   more tokens?
2. **Many people believe Bangla costs several times more than English.** Your measurement either
   supports that or contradicts it. If it contradicts it, where do you think the belief came from?
3. ShopWise's inbox is roughly 60% English, 30% Bangla, 10% pasted order JSON. Which of those three
   would you look at first if the monthly bill doubled?

### Part D — the number that actually bites

`count_tokens` measures what you *send*. Go back to section 4.7 of the session notebook, where a six-word
follow-up question cost several hundred input tokens.

Answer in one sentence: **for a support chat that runs 20 turns, which grows faster — the number of
messages, or the total tokens billed?** Say why.

> …

---
## ⭐⭐ Exercise 3 — Ticket loader

Open `warmup.py`. It ends at a `# Your turn` comment — this is that turn.

### Part A — count the tickets in each category

Build a dict mapping category → count, then print a small report. Use
`counts.get(key, 0) + 1`, which reads as *"whatever's there, or zero if nothing, plus one"*.

**Remember this shape.** In session 10 it is literally how a node in a graph updates state.

In [ ]:
counts: dict[str, int] = {}

for t in tickets:
    pass    # TODO: one line — add 1 to counts[t["category"]]

if not counts:
    print("counts is still empty — fill in the loop above.")
else:
    for category, n in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"{category:18} {'█' * n} {n}")

### Part B — who writes in the most?

Same pattern, different key: count tickets per customer (`from`). Which customer is most expensive to
support? Print them in descending order.

In [ ]:
by_customer: dict[str, int] = {}

# TODO: same pattern as Part A, keyed on t["from"]

print(by_customer or "TODO")

### Part C — put it back in `warmup.py`

Move your working code into `warmup.py`, under the `# Your turn` comment, as a function:

```python
def count_by(tickets: list[dict], field: str) -> dict[str, int]:
    ...
```

Then `count_by(tickets, "category")` and `count_by(tickets, "from")` both work from one function.
Run it with `uv run python warmup.py` and check the output still makes sense.

*Why this matters:* you just wrote a function that takes the **name of a field** as an argument
instead of hard-coding it. That's the difference between a script and a tool — and session 5 is
entirely about writing tools.

---
## ⭐⭐⭐ Exercise 4 — Survey the field

No code and no API calls — three browser tabs and about twenty minutes. Written answers.

Tonight you used one model from one company because a course needs stable ground, not because it is
the right answer. Choosing deliberately means knowing what else exists, what it costs, and who to
believe about it. That is a skill you will use in a design review long before you use it in code.

### Part A — leaderboards, read properly

**`arena.ai/leaderboard`** — open the overall board, then a category board (coding, or vision).

- Name one model that ranks **noticeably differently** between the two boards.
- Why might a model be strong at one and mediocre at the other?
- Section 6.3 gives you three caveats for reading these. Which of the three does your example
  illustrate?

> …

### Part B — the open half of the field

**`huggingface.co`** — search for something in your own language or domain (try `bangla`).

- Find one **model** and one **dataset**. Write down each name and its **licence**.
- Open a model card and name one limitation its authors admit to.
- Section 6.2 said "open weights" is not the same as "open source". Does the licence you found back
  that up?

> …

### Part C — what it would cost

**`openrouter.ai/models`** — the price list for hundreds of models in one place.
   - Find the cheapest model you'd consider for ticket triage, and the price per million tokens.
   - Compare it to a frontier model on the same page. **How many times more expensive is it?**
   - Using your section 5.2 measurement (roughly 100 tokens per triaged ticket), estimate the monthly cost of
     each at 200 tickets/day. Is the expensive one worth it *for this task*?

> …

That last question is the one you will be asked in a real job, and "we used the best model" is the
wrong answer. The right one names a task, a measurement and a budget.

---
## 💼 Interview questions

These are asked in real junior-AI-engineer interviews. Write your answer **before** reading the
sketches in `solutions.ipynb` — and write it as you'd actually say it out loud, not as bullet points.

### 1. A stakeholder asks: "why don't we just use the best model?"

They've seen a leaderboard. Talk them through how you'd actually choose a model for ShopWise's ticket
triage, in about a minute. You may not say "it depends" without immediately saying *on what*.

> …

### 2. Compute the cost of one call.

Your prompt is **2,000 tokens** and the answer is **500 tokens**. From this price sheet:

| | per 1M tokens |
|---|---|
| input | $0.10 |
| output | $0.40 |

- What does this single call cost?
- What does it cost for 5,000 calls a day, over a 30-day month?
- Your manager wants to halve that bill. **Which of the two numbers do you attack, and why?**

> …

### 3. Your client is a hospital. They ask whether their patient data will be safe.

Explain, without jargon, what the closed-weights vs open-weights choice means for them, and what you
would recommend. Mention at least one thing you would need to know before answering properly.

> …

---
## Checking yourself

`solutions.ipynb` has full workings plus answer sketches for the interview questions. Two rules:

1. **Write something first**, even if it's wrong. A wrong answer you wrote is worth more than a right
   one you read.
2. After reading a solution, **close it and re-type the code from memory.** If you can't, you haven't
   learned it yet — and that's useful information, not a failure.

Bring anything that stayed confusing to the start of Saturday's session. The first ten minutes are
for exactly that.